<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 강화학습·에이전트 · 09. AlphaZero / MCTS

## Mastering Chess and Shogi by Self-Play with a General Reinforcement Learning Algorithm

- **원 논문:** [Mastering Chess and Shogi by Self-Play with a General Reinforcement Learning Algorithm](https://arxiv.org/abs/1712.01815)
- **저자 / 발표:** David Silver et al. · arXiv / Science (2017)
- **난이도 / 예상 시간:** 고급 · 약 75분
- **선수 지식:** MCTS, policy/value network, self-play targets
- **로컬 학습 데이터:** `data/field_curriculum/gridworld.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 로컬 deterministic MDP에서 PUCT visit policy와 joint policy/value loss를 구현한다.

**축소하거나 생략한 부분:** 논문은 chess/shogi self-play와 대규모 residual network를 사용한다. 축소판은 exact model과 value-iteration leaf oracle로 MCTS/visit target/loss를 분리한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | Main text, network paragraph and Eq. (1) | (p,v)=f_theta(s) and joint loss | visit policy cross-entropy, outcome value MSE, L2를 결합한다. |
| Main text, MCTS paragraph | PUCT search and root visit counts | prior P, mean Q, visit N으로 action을 고르고 pi를 만든다. |
| Main text, self-play paragraph | search-improved policy targets | a_t~pi_t와 terminal z의 축소판으로 모든 state search targets를 만든다. |

## 핵심 재현

        네트워크 $(p,v)=f_\theta(s)$가 prior와 leaf value를 내고 MCTS root visit counts가
        개선된 policy $\pi$를 만듭니다. 마지막에는 Eq. (1)의 policy/value/L2 loss로 network를 학습합니다.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(491)
np.random.seed(491)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Tiny scalar/table loops may be faster on CPU even when a GPU exists;
# AI_LAB_DEVICE=cpu remains the low-overhead override.
DATA_PATH = Path("data/field_curriculum/gridworld.json")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
config = json.loads(DATA_PATH.read_text(encoding="utf-8"))
required = {
    "seed",
    "size",
    "start_state",
    "terminal_state",
    "gamma",
    "transitions",
    "bandit_means",
    "offline_action_noise",
}
assert required.issubset(config), f"JSON 키 불일치: {sorted(config)}"
rows = config["transitions"]
states = torch.tensor([int(r["state"]) for r in rows], dtype=torch.long)
actions = torch.tensor([int(r["action"]) for r in rows], dtype=torch.long)
rewards = torch.tensor([float(r["reward"]) for r in rows], dtype=torch.float32)
next_states = torch.tensor([int(r["next_state"]) for r in rows], dtype=torch.long)
dones = torch.tensor([float(r["done"]) for r in rows], dtype=torch.float32)
n_states = int(max(states.max(), next_states.max()).item()) + 1
n_actions = int(actions.max().item()) + 1
gamma = float(config["gamma"])
start_state, terminal_state = int(config["start_state"]), int(config["terminal_state"])
transition_table = {
    (int(r["state"]), int(r["action"])): (
        int(r["next_state"]),
        float(r["reward"]),
        bool(r["done"]),
    )
    for r in rows
}


def encode(s):
    return F.one_hot(torch.as_tensor(s, dtype=torch.long), n_states).float()


def encode_device(s):
    return encode(s).to(DEVICE)


def env_step(state, action):
    return transition_table[(int(state), int(action))]


def value_iteration(iterations=300):
    q = torch.zeros(n_states, n_actions)
    for _ in range(iterations):
        old = q.clone()
        for (s, a), (ns, r, done) in transition_table.items():
            q[s, a] = r if done else r + gamma * old[ns].max()
    return q


q_star = value_iteration()
assert torch.isfinite(q_star).all() and len(rows) > 0
print(ACCELERATOR.summary())
print(
    "RL environment, replay metadata, and tree search stay on CPU; "
    "neural batches use DEVICE."
)
print(
    f"local MDP: states={n_states}, actions={n_actions}, "
    f"transitions={len(rows)}, gamma={gamma}"
)

## 포트폴리오 구현 설계: 수식 → 에이전트 책임 → 검증

논문의 핵심 update를 실행 가능한 에이전트의 `forward`, `loss`, `update`,
`evaluate`로 나눕니다. 아래 식의 선택·평가·gradient 경계를 메서드 본문에서
한 줄씩 추적하세요.

$$
\mathcal L=(z-v)^2-\pi^\top\log p+c\lVert\theta\rVert_2^2,\qquad U(s,a)=c_{puct}P(s,a)\frac{\sqrt{\sum_bN(s,b)}}{1+N(s,a)}
$$

- **기호와 역할:** $p/v$는 network policy/value, $\pi$는 MCTS visit target, $P,Q,N$은 tree prior/mean value/visit count입니다.
- **shape/state 계약:** `state [B] → policy logits [B, A], value [B]; root P/Q/N/pi [A]`
- **논문 위치:** `Main text, network paragraph and Eq. (1)`의 **(p,v)=f_theta(s) and joint loss**
- **코드 Task:** PUCT selection, visit policy, policy-value-L2 loss와 search target을 구현합니다.
- **학습·평가 흐름:** network prior/value → MCTS simulations → visit pi → joint update → agreement 평가
- **원문 대비 한계:** 논문은 chess/shogi self-play와 대규모 residual network를 사용한다. 축소판은 exact model과 value-iteration leaf oracle로 MCTS/visit target/loss를 분리한다.
- **구현 이유:** target, gradient, optimizer 책임을 Agent 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 Bellman target, policy objective, search score 같은 핵심 계산을
불투명한 trainer 뒤에 감추지 않습니다. 실습본에서는 공개 API를 유지하면서 TODO를
채우고, 각 메서드의 입력 shape와 gradient가 끊기는 위치를 설명하세요.

In [ ]:
class AlphaZeroSearchAgent(nn.Module):
    """AlphaZero / MCTS의 핵심 학습·평가 경로. 입출력 계약: state [B] → policy logits [B, A],
    value [B]; root P/Q/N/pi [A]."""

    def __init__(
        self,
        policy_value_network,
        optimizer,
        transition_fn,
        leaf_action_values,
        discount,
        action_count,
        exploration=1.5,
    ):
        """network, optimizer, target 또는 table 상태를 명시적으로 저장한다."""
        super().__init__()
        self.policy_value_network = policy_value_network
        self.optimizer = optimizer
        self.transition_fn = transition_fn
        self.leaf_action_values = leaf_action_values.detach().cpu()
        self.discount = discount
        self.action_count = action_count
        self.exploration = exploration
        self.last_visit_count = torch.zeros(action_count)
        self.last_root_policy = torch.zeros(action_count)

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"action_count": self.action_count, "c_puct": self.exploration}

    def forward(self, state):
        """policy/value/Q 순전파를 계산한다. shape 계약: state [B] → policy logits [B, A], value
        [B]; root P/Q/N/pi [A]."""
        policy_logits, value = self.policy_value_network(state)
        return (torch.softmax(policy_logits, dim=1), value)

    def select_action(self, state, simulations=80, temperature=1.0):
        """PUCT simulation을 수행하고 temperature visit policy, count, prior와 행동을 반환한다."""
        if simulations < 1 or temperature < 0.0:
            raise ValueError("simulations must be positive and temperature nonnegative")
        with torch.no_grad():
            prior, _ = self.forward(torch.tensor([state]))
        prior = prior[0].detach().cpu()
        visit_count = torch.zeros(self.action_count)
        total_value = torch.zeros(self.action_count)
        for _ in range(simulations):
            mean_q = torch.where(
                visit_count > 0,
                total_value / visit_count.clamp_min(1.0),
                torch.zeros_like(total_value),
            )
            exploration_bonus = self.exploration * prior
            exploration_bonus = exploration_bonus * torch.sqrt(visit_count.sum() + 1.0)
            exploration_bonus = exploration_bonus / (1.0 + visit_count)
            action = int((mean_q + exploration_bonus).argmax())
            next_state, reward, done = self.transition_fn(state, action)
            leaf = reward
            if not done:
                leaf += self.discount * float(self.leaf_action_values[next_state].max())
            visit_count[action] += 1.0
            total_value[action] += leaf
        if temperature == 0.0:
            root_policy = torch.zeros_like(visit_count)
            root_policy[visit_count.argmax()] = 1.0
        else:
            scaled_visit = visit_count.pow(1.0 / temperature)
            root_policy = scaled_visit / scaled_visit.sum().clamp_min(1.0)
        self.last_visit_count = visit_count.clone()
        self.last_root_policy = root_policy.clone()
        action = int(root_policy.argmax())
        return (action, root_policy, visit_count, prior)

    def loss(self, state, target_policy, target_value):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        policy_logits, predicted_value = self.policy_value_network(state)
        policy_loss = (
            -(target_policy * torch.log_softmax(policy_logits, dim=1)).sum(dim=1).mean()
        )
        value_loss = F.mse_loss(predicted_value, target_value)
        regularization = sum(
            (
                parameter.square().sum()
                for parameter in self.policy_value_network.parameters()
            )
        )
        return policy_loss + value_loss + 0.0001 * regularization

    def update(self, state, target_policy, target_value):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        objective = self.loss(state, target_policy, target_value)
        self.optimizer.zero_grad()
        objective.backward()
        self.optimizer.step()
        return float(objective.detach())

    def evaluate(self, state, target_policy, target_value, visit_count, raw_prior):
        """no_grad 경로에서 정책 또는 value 품질 지표 dict를 반환한다."""
        with torch.no_grad():
            predicted_policy, predicted_value = self.forward(state)
            policy_cross_entropy = (
                -(target_policy * torch.log(predicted_policy.clamp_min(1e-08)))
                .sum(dim=1)
                .mean()
            )
            value_mse = F.mse_loss(predicted_value, target_value)
            improved_target_l1 = (target_policy - raw_prior).abs().mean()
            target_entropy = (
                -(target_policy * torch.log(target_policy.clamp_min(1e-08)))
                .sum(dim=1)
                .mean()
            )
            mean_visit_total = visit_count.sum(dim=1).float().mean()
        return {
            "policy_cross_entropy": float(policy_cross_entropy),
            "value_mse": float(value_mse),
            "improved_target_l1": float(improved_target_l1),
            "target_policy_entropy": float(target_entropy),
            "mean_visit_total": float(mean_visit_total),
        }

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `Main text, MCTS paragraph` — **PUCT search and root visit counts**
- **구현 이유:** prior P, mean Q, visit N으로 action을 고르고 pi를 만든다.
- **읽을 코드:** 아래 `implementation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
class PVNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(n_states, 32), nn.ReLU())
        self.policy = nn.Linear(32, n_actions)
        self.value = nn.Sequential(nn.Linear(32, 1), nn.Tanh())

    def forward(self, state):
        hidden = self.body(encode_device(state))
        return self.policy(hidden), self.value(hidden).squeeze(1)


net = PVNet().to(DEVICE)
opt = torch.optim.Adam(net.parameters(), lr=0.015)
portfolio_agent = AlphaZeroSearchAgent(
    net,
    opt,
    env_step,
    q_star,
    gamma,
    n_actions,
)
root_action, root_pi, root_visits, root_prior = portfolio_agent.select_action(
    start_state,
    simulations=80,
    temperature=1.0,
)
assert root_visits.sum() == 80
assert torch.allclose(root_pi.sum(), torch.tensor(1.0))
assert int(root_pi.argmax()) == root_action
assert (root_pi - root_prior).abs().sum() > 0.0

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `Main text, self-play paragraph` — **search-improved policy targets**
- **구현 이유:** a_t~pi_t와 terminal z의 축소판으로 모든 state search targets를 만든다.
- **읽을 코드:** 아래 `search` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
train_states_cpu = torch.tensor(
    [state for state in range(n_states) if state != terminal_state]
)
search_results = [
    portfolio_agent.select_action(
        int(state),
        simulations=80,
        temperature=1.0,
    )
    for state in train_states_cpu
]
search_actions = torch.tensor([result[0] for result in search_results])
target_pi_cpu = torch.stack([result[1] for result in search_results])
visit_counts_cpu = torch.stack([result[2] for result in search_results])
raw_priors_cpu = torch.stack([result[3] for result in search_results])
target_z_cpu = q_star[train_states_cpu].max(1).values
target_z_cpu = target_z_cpu / target_z_cpu.abs().max().clamp_min(1.0)
improved_target_l1 = float((target_pi_cpu - raw_priors_cpu).abs().mean())
assert torch.equal(
    visit_counts_cpu.sum(1), torch.full_like(visit_counts_cpu.sum(1), 80)
)
assert torch.equal(target_pi_cpu.argmax(1), search_actions)
assert torch.allclose(
    target_pi_cpu.sum(1),
    torch.ones_like(target_pi_cpu.sum(1)),
)
assert improved_target_l1 > 0.0

### 단계 4. 평가·시각화

- **논문의 어느 부분인가:** `Main text, network paragraph and Eq. (1)` — **(p,v)=f_theta(s) and joint loss**
- **구현 이유:** visit policy cross-entropy, outcome value MSE, L2를 결합한다.
- **읽을 코드:** 아래 `training-evaluation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
train_states, target_pi, target_z = ACCELERATOR.move(
    train_states_cpu,
    target_pi_cpu,
    target_z_cpu,
)
visit_counts, raw_priors = ACCELERATOR.move(
    visit_counts_cpu,
    raw_priors_cpu,
)
history = []
with torch.no_grad():
    l0, v0 = net(train_states)
    initial = float(
        -(target_pi * l0.log_softmax(1)).sum(1).mean() + F.mse_loss(v0, target_z)
    )

initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in net.parameters()]
)
for _ in range(200):
    history.append(portfolio_agent.update(train_states, target_pi, target_z))
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in net.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
portfolio_metrics = portfolio_agent.evaluate(
    train_states,
    target_pi,
    target_z,
    visit_counts,
    raw_priors,
)
with torch.no_grad():
    final_policy, _ = portfolio_agent.forward(train_states)
assert history[-1] < initial and parameter_delta > 0.0
assert torch.allclose(target_pi.sum(1), torch.ones_like(target_pi.sum(1)))
assert torch.equal(
    visit_counts.sum(1),
    torch.full_like(visit_counts.sum(1), 80),
)
assert portfolio_metrics["improved_target_l1"] > 0.0
assert portfolio_metrics["mean_visit_total"] == 80.0
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].bar(np.arange(n_actions), root_visits.detach().cpu())
axes[0].set_title("root visit counts")
axes[1].imshow(final_policy.detach().cpu().T, aspect="auto", vmin=0, vmax=1)
axes[1].set_title("learned p")
axes[2].plot(history)
axes[2].set_title("Eq. (1) loss")
fig.tight_layout()
plt.show()

search는 model rollouts와 value estimates를 여러 번 결합해 p보다 강한 정책을 만듭니다. pi를 다시 학습하면 network가 search의 개선을 amortize해 다음 search의 prior가 좋아집니다.

### 단계 5. 통합 Agent가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `Main text, network paragraph and Eq. (1)`의 **(p,v)=f_theta(s) and joint loss**
- **핵심 식:**

  $$
  \mathcal L=(z-v)^2-\pi^\top\log p+c\lVert\theta\rVert_2^2,\qquad U(s,a)=c_{puct}P(s,a)\frac{\sqrt{\sum_bN(s,b)}}{1+N(s,a)}
  $$

- **입출력 shape:** `state [B] → policy logits [B, A], value [B]; root P/Q/N/pi [A]`
- **구현 이유:** 앞 셀은 target, objective, search를 작은 연산으로
  분해해 확인하고, 이 셀은 같은 구성 요소를
  `AlphaZeroSearchAgent`에 주입해 최종 학습·평가 경로를
  하나로 묶습니다. visit policy cross-entropy, outcome value MSE, L2를 결합한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 정책·가치 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_agent.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_agent).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 chess/shogi self-play와 대규모 residual network를 사용한다. 축소판은 exact model과 value-iteration leaf oracle로 MCTS/visit target/loss를 분리한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.